minimum Required Parameters for searching through Fully Automated or Semi Automated Adopters that their execution env or test invocation is emulator related if exists

In [ ]:
# -*- coding: utf-8 -*-
"""
Emulator parameter extractor (DIY + GMD) with matrix-aware parsing (guardrails),
single-archive mode (All_Config_Files), and FA∪SA cohort filter.

Cohort:
  - RQ1_F1 == True
  - RQ1_F2 in {Fully_Auto, Semi_Auto}
  - execution_environment bucket in {DIY (emulator), GMD, Blank}
  - test_invocation bucket in {Gradle, ADB, Blank}

Extraction:
  - CI emulator "safe windows" (around emulator setup lines only)
  - GMD blocks in Gradle/KTS (apiLevel, systemImageSource, device, abi)
  - Matrix literals parsed and INCLUDED ONLY if ${ { matrix.KEY } } is referenced
    inside an emulator window in the same YAML. Stored in *_matrix_values (not counted
    as explicit values).

Output:
  - CSV: one row per repo with explicit params, matrix values, dataset API flags, provenance
"""

import os
import re
import json
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set, Tuple

# =========================
# PATHS (EDIT THESE)
# =========================
MAIN_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\5.0_Total_Repo.csv"
ARCHIVE_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files"  # <- single source of truth
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Params\Obs_4_Min_param_raw.csv"

# Single root (archive-only mode)
REPO_SEARCH_ROOTS = [ARCHIVE_DIR]

# =========================
# CONSTANTS / REGEX
# =========================

# CI lines that clearly indicate emulator setup/execution
EMULATOR_LINE = re.compile(
    r'(reactivecircus/android-emulator-runner|'
    r'\bavdmanager\b|'
    r'\bsdkmanager\b[^\n"]*system-images;android-\d+|'
    r'\bemulator\b\s+(-avd|@)|'
    r'android-wait-for-emulator|'
    r'circle-android\s+wait-for-boot)',
    re.I
)

# Parameter patterns (CI windows)
API_PATTERNS = [
    r'\bapi[-_ ]?level\s*:\s*([0-9]{2,3})\b',
    r'\bapiLevel\s*:\s*([0-9]{2,3})\b',
    r'system-images;android-([0-9]{2,3})\b',
]
SYSIMG_PATTERNS = [
    r'\btarget\s*:\s*(google_apis(?:_playstore)?)\b',
    r'system-images;android-\d{2,3};([a-z0-9_:-]+)\b',
    r'\bsystem[-_ ]?image(?:source)?\s*:\s*([A-Za-z0-9_\-:]+)\b',
]
ABI_PATTERNS = [
    r'\b(?:abi|arch)\s*:\s*(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
    r'system-images;android-\d{2,3};[a-z0-9_:-]+;(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
]
DEVICE_NAME_PATTERNS = [
    r'\bdevice\s*:\s*([A-Za-z0-9_ \-]+)\b',
    r'\bprofile\s*:\s*([A-Za-z0-9_ \-]+)\b',
    r'\b--device\s+"?([A-Za-z0-9_ \-]+)"?',
    r'\bavd[-_ ]?name\s*:\s*([A-Za-z0-9_ \-]+)\b',
]
PARAM_PATTERNS = {
    "api_level": API_PATTERNS,
    "system_image": SYSIMG_PATTERNS,
    "abi": ABI_PATTERNS,
    "device_name": DEVICE_NAME_PATTERNS,
}

# Variable tokens to record (not counted as explicit)
VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'
    r'\${{\s*(?:secrets|env|vars|inputs|matrix)\.[^}]+}}|'
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s)',
    re.I
)

# Gradle Managed Devices (GMD)
GMD_FILE_HIT = re.compile(r'\bmanagedDevices\b|\bManagedVirtualDevice\b', re.I)
GMD_BLOCK = re.compile(
    r'(managedDevices\s*\{[\s\S]{0,6000}?\})|(\(\s*ManagedVirtualDevice\s*\)\s*\{[\s\S]{0,4000}?\})',
    re.I
)
GMD_DEVICE = re.compile(r'\bdevice\s*=\s*["\']([^"\']+)["\']', re.I)
GMD_API    = re.compile(r'\bapiLevel\s*=\s*([0-9]{2,3})\b', re.I)
GMD_SYSIMG = re.compile(r'\bsystemImageSource\s*=\s*["\']([A-Za-z0-9_\-:]+)["\']', re.I)
GMD_ABI    = re.compile(r'\babi\s*=\s*["\'](x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)["\']', re.I)

FOLLOWABLE_EXTS = (".sh", ".bash", ".bat", ".cmd", ".ps1", ".json", ".yml", ".yaml")
GRADLE_FILES = ("build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts", "gradle.properties")

# Matrix mapping and references
MATRIX_KEY_MAP = {
    "api-level": "api_level",
    "apilevel": "api_level",
    "api_level": "api_level",
    "api": "api_level",
    "arch": "abi",
    "abi": "abi",
    "target": "system_image",
    "systemimage": "system_image",
    "system_image": "system_image",
    "systemimagesource": "system_image",
    "device": "device_name",
    "avd-name": "device_name",
    "avd_name": "device_name",
    "profile": "device_name",
}
MATRIX_REF_RE = re.compile(r'\${{\s*matrix\.([A-Za-z0-9_\-]+)\s*}}', re.I)

# =========================
# HELPERS
# =========================
def read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def windows_around_emulator(content: str, line_radius: int = 12) -> List[Tuple[int, int, str]]:
    lines = content.splitlines()
    hits = [i for i, l in enumerate(lines) if EMULATOR_LINE.search(l)]
    spans = []
    for i in hits:
        lo = max(0, i - line_radius)
        hi = min(len(lines), i + line_radius + 1)
        spans.append((lo, hi, "\n".join(lines[lo:hi])))
    return spans

def extract_params_from_segment(seg: str):
    vals: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS}
    toks: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS}
    for key, patterns in PARAM_PATTERNS.items():
        for pat in patterns:
            for m in re.finditer(pat, seg, flags=re.I | re.M):
                v = m.group(1) if (m.lastindex and m.group(1)) else m.group(0)
                v = (v or "").strip().strip('"').strip("'")
                if v and v.lower() not in ("api-level", "avd name"):
                    if v not in vals[key]:
                        vals[key].append(v)
    # tokens near param keywords
    for key in PARAM_PATTERNS.keys():
        window_re = re.compile(
            rf'(?i)(?:{key}|api[-_ ]?level|system[-_ ]?image|target|abi|arch|device|profile|avd[-_ ]?name)[^\n]{{0,160}}'
        )
        for w in window_re.finditer(seg):
            segment = seg[w.start():w.end()]
            for tm in VAR_TOKEN_RE.finditer(segment):
                tok = tm.group(0)
                if tok not in toks[key]:
                    toks[key].append(tok)
    return vals, toks

def collect_yaml_index(yaml_dir: Path) -> Dict[str, List[Path]]:
    """Index YAMLs in the archive by repo_key (owner.repo before '__')."""
    idx: Dict[str, List[Path]] = {}
    ydir = Path(yaml_dir)
    for p in ydir.iterdir():
        if p.is_file() and p.suffix.lower() in (".yml", ".yaml"):
            name = p.name.lower()
            if "__" in name:
                repo_key = name.split("__", 1)[0]  # owner.repo
                idx.setdefault(repo_key, []).append(p)
    return idx

def repo_key_from_full_name(full_name: str) -> str:
    return full_name.lower().replace("/", ".")

def find_file_refs_from_yaml(content: str) -> List[str]:
    refs: List[str] = []
    run_line = re.compile(r'(?mi)^\s*(?:run|script|command)\s*:\s*(.+)$')
    path_ref = re.compile(r'(?P<path>(?:\.{0,2}/|[A-Za-z]:\\)?[A-Za-z0-9._\-/\\]+(?:' + '|'.join([re.escape(e) for e in FOLLOWABLE_EXTS]) + r'))')
    for m in run_line.finditer(content):
        refs.extend(re.findall(path_ref, m.group(1)))
    for ref in re.findall(r'["\']([^"\']+\.(?:json|ya?ml|sh|bat|cmd|ps1))["\']', content, flags=re.I):
        refs.append(ref)
    out, seen = [], set()
    for r in refs:
        r_norm = r.strip().strip('"').strip("'")
        if r_norm not in seen:
            seen.add(r_norm); out.append(r_norm)
    return out

# -------- Archive-only resolvers --------
def find_repo_files(repo_key: str, roots: List[str], rel_path: str) -> List[Path]:
    """Resolve referenced files INSIDE the single archive by repo_key & path/basename."""
    results: List[Path] = []
    rel_norm = rel_path.replace("\\", "/").lstrip("./")
    base = Path(roots[0])

    # 1) exact suffix match under repo-key subtrees
    for p in base.rglob("*"):
        if repo_key in str(p).lower().replace("\\", "/"):
            if str(p).lower().replace("\\", "/").endswith("/" + rel_norm.lower()) and p.is_file():
                results.append(p)
    if results:
        return results

    # 2) basename match under repo-key subtrees
    bn = os.path.basename(rel_norm).lower()
    for p in base.rglob("*"):
        if repo_key in str(p).lower().replace("\\", "/") and p.is_file() and p.name.lower() == bn:
            results.append(p)
    if results:
        return results

    # 3) basename anywhere (last resort)
    for p in base.rglob(bn):
        if p.is_file():
            results.append(p)
    return results

def find_gradle_files(repo_key: str, roots: List[str]) -> List[Path]:
    """Find Gradle/KTS files for the repo inside the archive."""
    base = Path(roots[0])
    hits: List[Path] = []
    # Prefer repo-key subtrees
    for p in base.rglob("*"):
        if p.is_file() and p.name in GRADLE_FILES and repo_key in str(p).lower().replace("\\", "/"):
            hits.append(p)
    if hits:
        return hits
    # Fallback: any Gradle file with flat name starting with repo_key__
    for p in base.rglob(f"{repo_key}__*"):
        if p.is_file() and any(p.name.endswith(gf) for gf in GRADLE_FILES):
            hits.append(p)
    # Last resort: any gradle file anywhere
    if not hits:
        for gf in GRADLE_FILES:
            hits.extend(base.rglob(gf))
    return hits

# -------- GMD parsing --------
def extract_gmd_params(text: str) -> Dict[str, List[str]]:
    found = {"api_level": [], "system_image": [], "abi": [], "device_name": []}
    if not GMD_FILE_HIT.search(text):
        return found
    for tup in re.findall(GMD_BLOCK, text):
        block_txt = "".join(tup)
        for m in GMD_DEVICE.finditer(block_txt):
            v = m.group(1).strip()
            if v and v not in found["device_name"]: found["device_name"].append(v)
        for m in GMD_API.finditer(block_txt):
            v = m.group(1).strip()
            if v and v not in found["api_level"]: found["api_level"].append(v)
        for m in GMD_SYSIMG.finditer(block_txt):
            v = m.group(1).strip().lower()
            if v in ("google", "google_apis"): v = "google_apis"
            if v in ("playstore", "google_apis_playstore"): v = "google_apis_playstore"
            if v in ("aosp-atd", "aosp_atd"): v = "aosp-atd"
            if v and v not in found["system_image"]: found["system_image"].append(v)
        for m in GMD_ABI.finditer(block_txt):
            v = m.group(1).replace("_","-").lower()
            if v and v not in found["abi"]: found["abi"].append(v)
    return found

def finalize(values_list: List[str], tokens_list: List[str]) -> Tuple[str, str]:
    if values_list:
        return "; ".join(values_list), "explicit"
    if tokens_list:
        return "; ".join(tokens_list), "env_or_input"
    return "", "unspecified"

# -------- Cohort bucketing --------
def env_bucket(s: str) -> str:
    s0 = "" if pd.isna(s) else str(s)
    s = s0.strip().lower()
    if s == "" or s == "unknown":
        return "Blank"
    parts = [p.strip() for p in s.replace(";", ",").split(",") if p.strip()]
    if any("gmd" in p for p in parts): return "GMD"
    if any("emulator" in p for p in parts): return "DIY"
    return "Other"

def inv_bucket(s: str) -> str:
    s0 = "" if pd.isna(s) else str(s)
    s = s0.strip().lower()
    if s == "" or s == "unknown":
        return "Blank"
    if "adb" in s: return "ADB"
    if "gradle" in s: return "Gradle"
    return "Other"

def gather_api_levels_from_dataset(row: pd.Series) -> List[str]:
    levels: List[str] = []
    for c in row.index:
        m = re.fullmatch(r"api_(\d{2,3})", c, flags=re.I)
        if m:
            try:
                val = row[c]
                if pd.notna(val) and float(val) > 0:
                    levels.append(m.group(1))
            except Exception:
                pass
    return sorted(set(levels), key=lambda x: int(x))

# -------- Matrix parsing (guardrails) --------
def parse_matrix_literals(yaml_text: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {"api_level": [], "system_image": [], "abi": [], "device_name": []}
    try:
        data = yaml.safe_load(yaml_text)
    except Exception:
        return out
    if not isinstance(data, dict):
        return out

    def collect_from_matrix(matrix_obj):
        if not isinstance(matrix_obj, dict):
            return
        for k, v in matrix_obj.items():
            mapped = MATRIX_KEY_MAP.get(str(k).strip().lower())
            if not mapped:
                continue
            vals: List[str] = []
            if isinstance(v, list):
                vals = [str(x).strip() for x in v if isinstance(x, (str, int, float))]
            elif isinstance(v, (str, int, float)):
                vals = [str(v).strip()]
            # normalize system_image
            clean = []
            for t in vals:
                t_norm = t
                if mapped == "system_image":
                    tn = t.lower().replace(" ", "_")
                    if tn in ("google", "google_apis"): tn = "google_apis"
                    if tn in ("playstore", "google_apis_playstore"): tn = "google_apis_playstore"
                    if tn in ("aosp-atd", "aosp_atd"): tn = "aosp-atd"
                    t_norm = tn
                clean.append(t_norm)
            for c in clean:
                if c and c not in out[mapped]:
                    out[mapped].append(c)

    jobs = data.get("jobs", {})
    if isinstance(jobs, dict):
        for _, job in jobs.items():
            if not isinstance(job, dict): continue
            strat = job.get("strategy", {})
            if isinstance(strat, dict) and "matrix" in strat:
                collect_from_matrix(strat.get("matrix"))
    return out

def matrix_keys_referenced_in_emulator_windows(yaml_text: str) -> Set[str]:
    refs: Set[str] = set()
    for (lo, hi, win) in windows_around_emulator(yaml_text, line_radius=12):
        for m in MATRIX_REF_RE.finditer(win):
            refs.add(m.group(1).strip().lower())
    return refs

# =========================
# MAIN
# =========================
def main():
    # Load dataset
    df = pd.read_csv(MAIN_CSV, low_memory=False)
    df = lower_cols(df)

    env_col = "execution_environment_mr" if "execution_environment_mr" in df.columns else "execution_environment"
    for c in ["full_name", "rq1_f1", "rq1_f2", env_col]:
        if c not in df.columns:
            raise KeyError(f"Required column '{c}' not found in CSV.")

    # Buckets & cohort filter
    df["env_bucket"] = df[env_col].apply(env_bucket)
    df["inv_bucket"] = df.get("test_invocation", pd.Series([""]*len(df))).apply(inv_bucket)
    mask = (
        (df["rq1_f1"] == True) &
        (df["rq1_f2"].astype(str).isin(["Fully_Auto", "Semi_Auto"])) &
        (df["env_bucket"].isin(["DIY", "GMD", "Blank"])) &
        (df["inv_bucket"].isin(["Gradle", "ADB", "Blank"]))
    )
    cohort = df.loc[mask].copy()
    print(f"Cohort size (FA ∪ SA; Env DIY/GMD/Blank; Inv Gradle/ADB/Blank): {cohort['full_name'].nunique()} repos")

    # YAML index from the single archive
    ydir = Path(ARCHIVE_DIR)
    if not ydir.exists():
        raise FileNotFoundError(f"Archive directory not found: {ARCHIVE_DIR}")
    yaml_index = collect_yaml_index(ydir)

    rows = []
    for _, rec in cohort.iterrows():
        full_name = str(rec["full_name"]).strip()
        if not full_name:
            continue
        repo_key = repo_key_from_full_name(full_name)

        acc_vals: Dict[str, List[str]]   = {k: [] for k in PARAM_PATTERNS}
        acc_tokens: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS}
        provenance: Set[str] = set()

        # ---------- GMD PASS (Gradle/KTS in archive) ----------
        for gf in find_gradle_files(repo_key, REPO_SEARCH_ROOTS):
            gtxt = read_text(gf)
            if not gtxt: continue
            gmd_vals = extract_gmd_params(gtxt)
            if any(gmd_vals[k] for k in gmd_vals):
                provenance.add(f"{gf}::GMD")
            for k in ("api_level", "system_image", "abi", "device_name"):
                for v in gmd_vals[k]:
                    if v not in acc_vals[k]:
                        acc_vals[k].append(v)

        # ---------- YAML PASS (archive) ----------
        matrix_vals_eligible: Dict[str, Set[str]] = {
            "api_level": set(), "system_image": set(), "abi": set(), "device_name": set()
        }

        for yml in yaml_index.get(repo_key, []):
            ytxt = read_text(yml)
            if not ytxt:
                continue

            # (A) raw matrix literals in this YAML
            mat_lits = parse_matrix_literals(ytxt)

            # (B) which matrix keys are referenced in emulator windows (guardrail #1)
            referenced_keys = matrix_keys_referenced_in_emulator_windows(ytxt)

            # (C) explicit literals & tokens from emulator windows
            for (lo, hi, win) in windows_around_emulator(ytxt, line_radius=12):
                vals, toks = extract_params_from_segment(win)

                # avoid ndk abiFilters noise unless emulator line present
                if re.search(r'\bandroid\b.*\bndk\b.*\babiFilters\b', win, re.I | re.S) and not EMULATOR_LINE.search(win):
                    vals["abi"].clear()

                for k in PARAM_PATTERNS.keys():
                    for v in vals[k]:
                        if v not in acc_vals[k]:
                            acc_vals[k].append(v)
                    for t in toks[k]:
                        if t not in acc_tokens[k]:
                            acc_tokens[k].append(t)
                provenance.add(f"{yml.name}::lines {lo+1}-{hi}")

            # (D) Guarded merge of matrix literals (only if key seen in emulator windows)
            for mk, bucket in MATRIX_KEY_MAP.items():
                if mk in referenced_keys:
                    for v in mat_lits.get(bucket, []):
                        matrix_vals_eligible[bucket].add(v)
                    if referenced_keys:
                        provenance.add(f"{yml.name}::matrix_keys_used_in_emulator={sorted(list(referenced_keys))}")

            # (E) Follow referenced files INSIDE ARCHIVE and extract in windows too
            for ref in find_file_refs_from_yaml(ytxt):
                for base in REPO_SEARCH_ROOTS:
                    for rp in find_repo_files(repo_key, [base], ref):
                        rtxt = read_text(rp)
                        if not rtxt: continue
                        payload = rtxt
                        if rp.suffix.lower() == ".json":
                            try: payload = json.dumps(json.loads(rtxt))
                            except Exception: payload = rtxt
                        elif rp.suffix.lower() in (".yml", ".yaml"):
                            try: payload = json.dumps(yaml.safe_load(rtxt), default=str)
                            except Exception: payload = rtxt

                        for (lo, hi, win) in windows_around_emulator(payload, line_radius=12):
                            vals2, toks2 = extract_params_from_segment(win)
                            if re.search(r'\bandroid\b.*\bndk\b.*\babiFilters\b', win, re.I | re.S) and not EMULATOR_LINE.search(win):
                                vals2["abi"].clear()
                            for k in PARAM_PATTERNS.keys():
                                for v in vals2[k]:
                                    if v not in acc_vals[k]:
                                        acc_vals[k].append(v)
                                for t in toks2[k]:
                                    if t not in acc_tokens[k]:
                                        acc_tokens[k].append(t)
                            provenance.add(f"{rp.name}::lines {lo+1}-{hi}")

        # finalize explicit vs tokens
        api_level, api_status       = finalize(acc_vals["api_level"],    acc_tokens["api_level"])
        system_image, sysimg_status = finalize(acc_vals["system_image"], acc_tokens["system_image"])
        abi, abi_status             = finalize(acc_vals["abi"],          acc_tokens["abi"])
        device_name, dev_status     = finalize(acc_vals["device_name"],  acc_tokens["device_name"])

        # dataset API flags
        api_levels_ds = gather_api_levels_from_dataset(rec)
        api_level_total = rec["api_level_total"] if "api_level_total" in rec.index else ""

        # Row
        rows.append({
            "full_name": str(rec["full_name"]),
            "automation_level": str(rec["rq1_f2"]),
            "env_bucket": str(rec["env_bucket"]),
            "inv_bucket": str(rec["inv_bucket"]),

            # explicit values (from emulator windows / GMD)
            "api_level_from_files": api_level,        "api_level_from_files_status": api_status,
            "system_image": system_image,             "system_image_status": sysimg_status,
            "abi": abi,                               "abi_status": abi_status,
            "device_name": device_name,               "device_name_status": dev_status,

            # matrix literals (guarded) — separate channel
            "api_level_matrix_values":    "; ".join(sorted(matrix_vals_eligible["api_level"]))    if matrix_vals_eligible["api_level"]    else "",
            "system_image_matrix_values": "; ".join(sorted(matrix_vals_eligible["system_image"])) if matrix_vals_eligible["system_image"] else "",
            "abi_matrix_values":          "; ".join(sorted(matrix_vals_eligible["abi"]))          if matrix_vals_eligible["abi"]          else "",
            "device_name_matrix_values":  "; ".join(sorted(matrix_vals_eligible["device_name"]))  if matrix_vals_eligible["device_name"]  else "",

            # dataset-derived
            "api_levels_from_dataset": "; ".join(api_levels_ds) if api_levels_ds else "",
            "api_level_total": api_level_total,

            "sources": "; ".join(sorted(provenance)) if provenance else "",
        })

    out_df = pd.DataFrame(rows).sort_values(["automation_level","full_name"]).reset_index(drop=True)
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\nSaved: {OUTPUT_CSV}  (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(12).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv  (rows=333)
                     full_name emulator_label  adopter  emulator_keyword_seen                       api_level_from_files api_level_from_files_status                                                                                                   system_image system_image_status          abi   abi_status  device_name device_name_status api_levels_from_dataset  api_level_total                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [4]:
# -*- coding: utf-8 -*-
"""
Group emulator parameters by repo (full_name) and export a tidy table.

Input:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv

Output:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.2_Emulator_Params_Grouped.csv

Behavior:
- Groups by full_name
- Builds 4 columns with sorted, de-duplicated, comma-separated values:
    API_Level, System_Image, ABI, Device_Name
- Uses only *explicit* values per the `*_status` columns
- API_Level merges `api_level_from_files` (explicit only) + `api_levels_from_dataset`
"""

from pathlib import Path
import re
import pandas as pd

# -------- paths --------
BASE = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters")
INPUT_CSV  = BASE / "5.1_Emulator_Params_By_Repo.csv"
OUTPUT_CSV = BASE / "5.2_Emulator_Params_Grouped.csv"

# Optional fallback if running outside Windows (comment out if not needed)
FALLBACK = Path("/mnt/data/5.1_Emulator_Params_By_Repo.csv")
if not INPUT_CSV.exists() and FALLBACK.exists():
    INPUT_CSV = FALLBACK

# -------- helpers --------
SEP_RE = re.compile(r"[;,|]")  # split on semicolon, comma, or pipe

def split_values(s: str):
    """Split a cell into tokens by common separators; trim; drop empties."""
    if pd.isna(s):
        return []
    parts = [p.strip().strip('"').strip("'") for p in SEP_RE.split(str(s))]
    return [p for p in parts if p]

def dedup_preserve_order(items, key=None):
    """De-duplicate while preserving first occurrence order."""
    seen = set()
    out = []
    for x in items:
        k = key(x) if key else x
        if k not in seen:
            seen.add(k); out.append(x)
    return out

def numeric_sortable(x: str):
    """Return (int) for numeric tokens; fallback to large value to push non-numeric to end."""
    m = re.fullmatch(r"\d{1,3}", x.strip())
    return int(m.group(0)) if m else 10**9

def collect_param_from_rows(series_vals: pd.Series, series_status: pd.Series | None, only_explicit=True):
    """Collect explicit values from a pair (values, status)."""
    vals = []
    for v, st in zip(series_vals, (series_status if series_status is not None else [None] * len(series_vals))):
        if only_explicit and series_status is not None and str(st).strip().lower() != "explicit":
            continue
        vals.extend(split_values(v))
    return vals

# -------- load --------
df = pd.read_csv(INPUT_CSV)

# Be flexible with column name casing
cols = {c.lower(): c for c in df.columns}
def c(name):  # map a desired lower-name to actual column if present
    return cols.get(name.lower())

required = ["full_name", "api_level_from_files", "api_level_from_files_status",
            "system_image", "system_image_status", "abi", "abi_status",
            "device_name", "device_name_status", "api_levels_from_dataset"]
missing = [r for r in required if c(r) is None]
if missing:
    # We can still proceed if some optional columns are missing; only 'full_name' is mandatory.
    if c("full_name") is None:
        raise KeyError(f"Missing required column 'full_name' in {INPUT_CSV}")
    # Warn (print) but continue
    print("Warning: missing columns:", missing)

# -------- group & aggregate --------
grouped_rows = []
for full_name, g in df.groupby(df[c("full_name")]):
    # API levels from files (explicit only)
    api_from_files = collect_param_from_rows(
        g[c("api_level_from_files")] if c("api_level_from_files") else pd.Series(dtype=str),
        g[c("api_level_from_files_status")] if c("api_level_from_files_status") else None,
        only_explicit=True
    )
    # API levels from dataset (no status column)
    api_from_ds = []
    if c("api_levels_from_dataset"):
        for cell in g[c("api_levels_from_dataset")]:
            api_from_ds.extend(split_values(cell))

    # Combine & clean API levels (keep only numeric-looking tokens)
    api_all = [t for t in api_from_files + api_from_ds if re.fullmatch(r"\d{1,3}", str(t).strip())]
    api_all = dedup_preserve_order(api_all, key=lambda x: x.strip())
    api_all_sorted = sorted(api_all, key=numeric_sortable)
    api_str = ", ".join(api_all_sorted)

    # System image (explicit only)
    sysimg_vals = collect_param_from_rows(
        g[c("system_image")] if c("system_image") else pd.Series(dtype=str),
        g[c("system_image_status")] if c("system_image_status") else None,
        only_explicit=True
    )
    sysimg_vals = dedup_preserve_order(sysimg_vals, key=lambda x: x.lower())
    sysimg_vals_sorted = sorted(sysimg_vals, key=lambda x: x.lower())
    sysimg_str = ", ".join(sysimg_vals_sorted)

    # ABI (explicit only)
    abi_vals = collect_param_from_rows(
        g[c("abi")] if c("abi") else pd.Series(dtype=str),
        g[c("abi_status")] if c("abi_status") else None,
        only_explicit=True
    )
    abi_vals = dedup_preserve_order(abi_vals, key=lambda x: x.lower())
    abi_vals_sorted = sorted(abi_vals, key=lambda x: x.lower())
    abi_str = ", ".join(abi_vals_sorted)

    # Device name (explicit only)
    dev_vals = collect_param_from_rows(
        g[c("device_name")] if c("device_name") else pd.Series(dtype=str),
        g[c("device_name_status")] if c("device_name_status") else None,
        only_explicit=True
    )
    dev_vals = dedup_preserve_order(dev_vals, key=lambda x: x.lower())
    dev_vals_sorted = sorted(dev_vals, key=lambda x: x.lower())
    dev_str = ", ".join(dev_vals_sorted)

    grouped_rows.append({
        "full_name": full_name,
        "API_Level": api_str,
        "System_Image": sysimg_str,
        "ABI": abi_str,
        "Device_Name": dev_str,
    })

out = pd.DataFrame(grouped_rows).sort_values("full_name").reset_index(drop=True)

# -------- save --------
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_CSV, index=False)

print(f"Read from: {INPUT_CSV}")
print(f"Wrote grouped table: {OUTPUT_CSV}")
print(f"Rows: {len(out)}")
print(out.head(12).to_string(index=False))


Read from: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv
Wrote grouped table: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.2_Emulator_Params_Grouped.csv
Rows: 333
                     full_name          API_Level System_Image    ABI Device_Name
              4ertuk.audioview                 25                                
               a-mabe.openhiit             34, 35              x86_64 pixel_6_pro
a914-gowtham.compose-ratingbar         19, 22, 30                                
       aakira.expandablelayout                 21                                
  abdelaziz-mahdy.pytorch_lite                 33              x86_64     Nexus 6
             ably.ably-flutter             24, 29                                
               achep.acdisplay                 23                                
      activitywatch.aw-android                     google_apis x86_64 